# MCP Server

**Prerequisites:** `01_mcp_concepts.ipynb`, `02_mcp_client.ipynb`. From `03_tools`: all five, particularly `04_verification.ipynb`

Everything so far was written from the consuming side, where a server's behaviour is a fact to work around. This notebook turns it over.

Doing it in this order is the point. We now know precisely what a client wishes a server would do, because notebook 2 was spent building workarounds for servers that did not do it — inferring a failure kind from one bool, putting back an `additionalProperties` the schema omitted, guessing whether a tool was safe to call twice. Every one of those workarounds names a thing a server could simply have said.

So the question here is not "how do I write an MCP server", which is four lines. It is: **what does a good server owe a client, and how much of `03_tools` can be made to survive the wire if the server is trying?**

The answer turns out to be *almost all of it*, including one thing notebook 2 declared structurally impossible.

## Setup

Two servers to compare. `server.py` is the naive one from notebook 1 — tools declared with `@server.tool`, schemas generated by the SDK. `registry_server.py` is an ordinary `03_tools` `Registry` put through `expose()`.

In [1]:
import json
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
sys.path.append(str(Path.cwd().parent.parent / "01_foundations"))
sys.path.append(str(Path.cwd().parent.parent / "03_tools"))

from invoke import invoke

from client import SERVER
from remote import remote_tools
from session import Session

REGISTRY_SERVER = Path.cwd().parent / "registry_server.py"

naive = Session(SERVER).__enter__()
ours = Session(REGISTRY_SERVER).__enter__()

print(naive.server_name, "->", [t.name for t in naive.list_tools()])
print(ours.server_name, "->", [t.name for t in ours.list_tools()])

forge-demo -> ['weather', 'echo', 'whoami', 'raises', 'described', 'detailed']
forge-registry -> ['weather', 'pressure', 'save_note']


## What the registry server is

Nothing about it is MCP-shaped. `registry_server.py` is chapter 3's tools, written the way chapter 3 wrote them:

```python
@tool
def weather(city: str, units: str = "celsius") -> str:
    """Look up the current weather in a city."""
    reading = READINGS.get(city.lower())
    if reading is None:
        raise ToolError(f"No weather station for {city}. Try a larger nearby city.")
    return f"{reading} degrees {units}, clear skies in {city}."

registry = Registry([weather, pressure, verified(save_note, note_was_saved_whole)])
server = expose(registry, name="forge-registry", timeout=10, max_tokens=400)
```

Three tools: one that works and can refuse, one with a `KeyError` in it, and one that lies about having saved a whole note — wrapped in `verified()`, which checks the store afterwards. `expose()` is the only line that has heard of MCP.

## The schema, and where it comes from

Notebook 1 held chapter 3's `weather` schema next to the same tool's MCP schema and found two differences. Here are both servers describing their own `weather`:

In [2]:
def schema_of(session, name="weather"):
    return next(t for t in session.list_tools() if t.name == name).input_schema

print("generated by the SDK from a function:")
print(" ", json.dumps(schema_of(naive), sort_keys=True))
print()
print("built by Tool.schema and published as-is:")
print(" ", json.dumps(schema_of(ours), sort_keys=True))

generated by the SDK from a function:
  {"properties": {"city": {"title": "City", "type": "string"}, "units": {"default": "celsius", "title": "Units", "type": "string"}}, "required": ["city"], "title": "weatherArguments", "type": "object"}

built by Tool.schema and published as-is:
  {"additionalProperties": false, "properties": {"city": {"type": "string"}, "units": {"default": "celsius", "type": "string"}}, "required": ["city"], "type": "object"}


Both defects gone. No `title` keys repeating the field names, and `additionalProperties: false` is there, so a client sending `citty` is told rather than ignored.

That did not require writing a schema by hand — it is `Tool.schema`, the same property chapter 3 built, published unchanged. Which is the part worth keeping: the description on the wire is read off the function's signature, so it **cannot drift from the function**. That was `03_tools` notebook 1's entire opening argument, and it holds across a process boundary for free.

## One check, in the place that can classify it

There is a subtlety in `expose()` worth pulling out, because it looks backwards.

`MCPServer` builds its own Pydantic model from the handler's signature and validates against it *before* the handler runs. If the handler had the real signature, every bad call would be rejected there — with an SDK-shaped error, before `invoke()` ever saw it. The caller would get a string, and none of chapter 3's classification would happen.

So `expose()` gives the handler a **deliberately permissive** signature — every parameter optional, typed `Any` — while advertising the strict schema. The gate in front of the handler is wide open; the real check is one layer further in, where it can produce a `Result` with a `kind`:

In [3]:
r = ours.call("weather", {"city": 42})

print("is_error:", r.is_error)
print("text:    ", r.content[0].text.splitlines()[0])
print("kind:    ", r.meta.get("forge/kind"))

is_error: True
text:     1 validation error for weather_args
kind:     bad_arguments


`bad_arguments`, from a server. Notebook 2 got that kind by validating locally before sending — which is still the right thing to do, and still means this path is rarely taken. But a client that does *not* validate now gets a correctly classified answer instead of a generic one.

**The advertised schema and the internal check are different things.** The schema is strict because it is what the model reads. The gate is open because validation belongs where the taxonomy lives.

## Every ending, across the wire

Here is what the server actually puts on the wire for each kind of failure.

In [4]:
calls = [
    ("weather",   {"city": "Oslo"}),
    ("weather",   {"city": "Vestby"}),
    ("pressure",  {"city": "Vestby"}),
    ("save_note", {"name": "n1", "text": "x" * 40}),
]

for name, args in calls:
    r = ours.call(name, args)
    forge = {k.split("/", 1)[1]: v for k, v in (r.meta or {}).items() if k.startswith("forge/")}
    print(f"{name}({json.dumps(args)})")
    print(f"   is_error={r.is_error}  content={r.content[0].text[:58]!r}")
    print(f"   _meta   {forge}")

weather({"city": "Oslo"})
   is_error=False  content='4 degrees celsius, clear skies in Oslo.'
   _meta   {'kind': 'ok', 'repeatable': True, 'truncated': False}
weather({"city": "Vestby"})
   is_error=True  content='No weather station for Vestby. Try a larger nearby city.'
   _meta   {'kind': 'refused', 'repeatable': True, 'truncated': False}
pressure({"city": "Vestby"})
   is_error=True  content='pressure could not be completed.'
   _meta   {'kind': 'broken', 'repeatable': True, 'truncated': False, 'cause': "KeyError: 'Vestby'"}
save_note({"name": "n1", "text": "xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"})
   is_error=True  content="'n1' holds 20 of 40 characters."
   _meta   {'kind': 'unverified', 'repeatable': False, 'truncated': False}


Look at `pressure`. The content the model may read is `pressure could not be completed.` — and the actual `KeyError` is in `_meta`, which no model sees.

**That is the thing notebook 1 found no server doing, and it could not be done from the client side.** Only the server knows which of its failures was a bug. `expose()` reads `Result.for_the_model`, which `invoke()` computed, and splits the message on it — the same rule `invoker()` applies locally, applied here because the conversation is in another process and cannot apply it itself.

## The round trip

Now hand those back to notebook 2's client, unchanged.

In [5]:
tools = {t.name: t for t in remote_tools(ours)}

for name, args in calls + [("weather", {"city": 42})]:
    r = invoke(tools[name], args, timeout=10)
    print(f"{name:10} kind={r.kind:14} for_the_model={r.for_the_model!s:5} "
          f"safe_to_retry={r.safe_to_retry!s:5}")
    print(f"           {r.content.splitlines()[0][:72]!r}")

weather    kind=ok             for_the_model=True  safe_to_retry=False
           '4 degrees celsius, clear skies in Oslo.'
weather    kind=refused        for_the_model=True  safe_to_retry=True 
           'No weather station for Vestby. Try a larger nearby city.'
pressure   kind=broken         for_the_model=False safe_to_retry=False
           "pressure is broken: RuntimeError: KeyError: 'Vestby'"
save_note  kind=unverified     for_the_model=True  safe_to_retry=False
           "'n1' holds 20 of 40 characters."
weather    kind=bad_arguments  for_the_model=True  safe_to_retry=True 
           '1 validation error for weatherArguments'


Five kinds, all correct, across a process boundary — and `remote.py` was not modified to make this work. It already read `forge/*`; notebook 2 just never had a server that sent them.

Set that against notebook 2's table, where the same client talking to a silent server recovered four kinds and guessed the rest:

| | foreign server (nb 2) | our server (here) |
|---|---|---|
| `ok` | yes | yes |
| `bad_arguments` | yes, by validating first | yes, either side |
| `timeout` | yes, the client's clock | yes, and the server's too |
| `broken` vs `refused` | **guessed** — defaults to `refused` | **exact**, and the cause travels in `_meta` |
| `unverified` | declared impossible | see below |

## The one notebook 2 got wrong

Notebook 2 listed `unverified` as structurally impossible, on the grounds that `verified()` cannot inspect another process's world. That reasoning was right and the conclusion was wrong, because it assumed the *client* had to do the verifying.

In [6]:
r = invoke(tools["save_note"], {"name": "n2", "text": "y" * 40})

print("kind:         ", r.kind)
print("content:      ", r.content)
print("safe_to_retry:", r.safe_to_retry)

kind:          unverified
content:       'n2' holds 20 of 40 characters.
safe_to_retry: False


`unverified`, over a wire.

`save_note` claims to have saved forty characters and actually stores twenty. The check that catches it reads the store — not the tool's own report, which says it worked — and the store is on the server. So verification does not merely *survive* the boundary; **the server is the only place it can happen at all.** The client has no access to the world being checked.

That generalises past this chapter. `verified()` asks "did the thing actually happen?", and that question can only be answered where the thing was supposed to happen. A tool that crosses a process boundary should carry its own verification with it, because nobody downstream is in a position to do it.

And notice `safe_to_retry` is False. `save_note` was declared `@tool(repeatable=False)`, and that fact also made the trip:

In [7]:
for t in ours.list_tools():
    hints = t.annotations
    print(f"{t.name:10} idempotentHint={hints.idempotent_hint}")

print()
for t in remote_tools(ours):
    print(f"{t.name:10} Tool.repeatable={t.repeatable}")

weather    idempotentHint=True
pressure   idempotentHint=True
save_note  idempotentHint=False

weather    Tool.repeatable=True
pressure   Tool.repeatable=True
save_note  Tool.repeatable=False


This one deliberately does **not** use `forge/`. MCP has `idempotentHint` and `readOnlyHint` for exactly this, and two things follow from using the standard field instead of inventing one.

It arrives at **discovery** rather than at call time, which is when the client builds its `Tool` and has to decide — a `forge/repeatable` on a result would arrive after the call it was needed for. And any client reads it, not only ours.

That is the general rule for this whole exercise: **say it in the protocol where the protocol has a word for it, and in `_meta` only where it does not.**

## A registry is not an export list

`expose()` publishes nothing unless asked. That is not caution for its own sake.

A `Registry` answers "which tools may *this agent* use". A server answers "which tools may *anyone who connects* use". Those are different questions, and `03_tools`' file tools are the clearest case: `file_tools(root)` is confined to a folder on this machine, which is a sentence about this machine — not a promise to a stranger who has a different idea of which folder matters.

In [8]:
from expose import expose
from registry import Registry
from tools import tool

@tool
def public_lookup(term: str) -> str:
    """Something safe to hand anybody."""
    return f"definition of {term}"

@tool(name="delete_everything")
def dangerous(confirm: bool) -> str:
    """Something that should never leave this process."""
    return "gone"

internal = Registry([public_lookup, dangerous])

everything = expose(internal, name="oops")
careful = expose(internal, name="careful", names=["public_lookup"])

print("the whole registry: ", sorted(everything._tool_manager._tools))
print("what we chose:      ", sorted(careful._tool_manager._tools))

the whole registry:  ['delete_everything', 'public_lookup']
what we chose:       ['public_lookup']


`subset()` was written in `03_tools` to keep a token bill down. Here it is doing something else entirely — it is the export boundary, and the fact that the same method serves both is a coincidence worth not relying on. Publishing deserves its own deliberate list, because the cost of getting it wrong is not a larger prompt.

## What a stranger gets

One honest limit. Everything above works because both ends agree about `forge/*`, and `forge/*` is our convention, not MCP's. A client that has never heard of it sees this:

In [9]:
r = ours.call("pressure", {"city": "Vestby"})

print("what any MCP client sees:")
print("   is_error:", r.is_error)
print("   content: ", r.content[0].text)
print()
print("what it ignores:")
print("   ", {k: v for k, v in r.meta.items() if k.startswith("forge/")})

what any MCP client sees:
   is_error: True
   content:  pressure could not be completed.

what it ignores:
    {'forge/kind': 'broken', 'forge/repeatable': True, 'forge/truncated': False, 'forge/cause': "KeyError: 'Vestby'"}


A stranger gets `is_error: True` and `pressure could not be completed.` — correct, safe, and less useful than what we get. It loses the kind and the cause, and it has no way to know it is missing them.

That is the right shape for an extension: **nothing a stranger reads is wrong, and nothing they cannot read is load-bearing.** The content is still honest on its own; `_meta` only adds precision. A server that put the actual message in `_meta` and a placeholder in `content` would be broken for everyone but itself.

It is also a reminder of what the chapter has and has not shown. Notebook 2's inference exists because most servers are strangers, and building a good one does not change that — it only means the servers *you* write stop being part of the problem.

In [10]:
naive.__exit__(None, None, None)
ours.__exit__(None, None, None)
print("both sessions closed")

both sessions closed


## Where this leaves the chapter

Nearly all of `03_tools` survives a process boundary, provided the server is trying:

| | crosses? |
|---|---|
| a schema that cannot drift from its function | yes — `Tool.schema` published as-is |
| `extra="forbid"` and no `title` noise | yes |
| the six-ending taxonomy | yes, in `_meta` |
| `for_the_model` hiding a bug from the model | yes, and only the server can do it |
| `safe_to_retry` | yes, via MCP's own `idempotentHint` |
| `verified()` | yes — and the server is the **only** place it works |
| a timeout | yes, but now there are two, and neither knows about the other |

The last row is the one this notebook has not dealt with. `expose(timeout=10)` is the server's limit, `Client(read_timeout_seconds=)` is the client's, and a call can exceed one, the other, or both, with different results each way. That is a transport question, and `05_mcp_transport.ipynb` is next — along with the thing that matters more than any of this: a stdio server we launched inherits our environment, including every key in it, so the process boundary and the trust boundary are **not** the same line.